In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Simulate High-Frequency Market Data
# In a real TS demo, you'd use a large-scale data pipeline.
np.random.seed(42)
days = 252
market_returns = np.random.normal(0.0005, 0.01, days)
# NVDA has a Beta of ~1.8 (very sensitive to market)
beta_nvda = 1.8
# The 'Residual' is the part of the move NOT explained by the market
residual_returns = np.random.normal(0, 0.005, days)
nvda_returns = (beta_nvda * market_returns) + residual_returns

# 2. Simulate an Alpha Signal (e.g., Sentiment Score)
# We'll make the signal slightly predictive of the RESIDUAL, not the market.
sentiment_signal = residual_returns + np.random.normal(0, 0.002, days)

df = pd.DataFrame({
    'Market': market_returns,
    'NVDA': nvda_returns,
    'Sentiment': sentiment_signal
})

# 3. The Two Sigma Move: Extracting the Residual
# We perform an OLS regression to find the pure Alpha signal.
X = sm.add_constant(df['Market']) # Market is the "Beta" factor
model = sm.OLS(df['NVDA'], X).fit()
df['Residual'] = model.resid  # This is NVDA's return minus the market effect

# 4. Validate the Signal
# Does our sentiment actually predict the residual?
signal_check = sm.OLS(df['Residual'], sm.add_constant(df['Sentiment'])).fit()

print("--- Two Sigma Alpha Extract Report ---")
print(f"Calculated Beta: {model.params['Market']:.4f}")
print(f"Signal Information Coefficient (IC): {df['Sentiment'].corr(df['Residual']):.4f}")
print(f"Signal P-Value: {signal_check.pvalues['Sentiment']:.4f}")

if signal_check.pvalues['Sentiment'] < 0.05:
    print("✅ ALPHA DETECTED: Signal is statistically significant.")
else:
    print("❌ NOISE: Signal failed the significance test.")

--- Two Sigma Alpha Extract Report ---
Calculated Beta: 1.8115
Signal Information Coefficient (IC): 0.9236
Signal P-Value: 0.0000
✅ ALPHA DETECTED: Signal is statistically significant.
